## Making a football fair team splitter

In [302]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler

In [303]:
df = pd.read_csv('zoki_liga.csv')

In [304]:
df.head(10)

,index,name,position,matches,goals,assists,goal_contirbution,gc_per_match,hat-trick,hat-trick-assists,...,losses,loss_percentage,motm,nominations,potm,gotm,passing,physical,gk_ability,skill
0,1,Daco,G,25,4,7,11,0.44,0,0,...,12,48%,0,1,0,0,3.5,5.5,7.5,1.50
1,2,Leo,D,23,30,15,45,1.96,3,3,...,9,39%,3,1,2,0,7.0,8.5,0.0,4.00
2,3,Simikj,M,23,8,19,27,1.17,0,1,...,12,52%,0,1,0,0,6.5,3.5,0.0,4.25
3,4,Stef,A,22,17,10,27,1.23,2,0,...,10,45%,1,2,0,0,5.0,6.0,0.0,2.50
4,5,Neno,G,22,0,6,6,0.27,0,0,...,9,41%,0,2,0,0,5.0,7.5,6.5,2.00
5,6,Premcho,A,19,26,16,42,2.21,5,0,...,11,58%,1,1,0,2,7.5,7.0,0.0,4.50
6,7,Hito,A,18,48,15,63,3.50,10,2,...,6,33%,4,2,1,2,7.0,8.0,0.0,4.50
7,8,Borjan,M,17,21,12,33,1.90,3,0,...,6,35%,0,0,0,0,5.0,6.5,0.0,1.50
8,9,Ancho,D,17,2,0,2,0.12,0,0,...,7,41%,0,0,0,0,1.5,3.5,0.0,1.00
9,10,Joker,A,15,18,13,31,2.07,0,1,...,4,27%,1,0,0,0,6.5,4.0,0.0,3.50


In [305]:
df = df.drop(columns=['index','matches','goal_contirbution','owngoals','freekicks','penalty_saves','penalty_stats','w','losses','nominations','gotm'])

In [306]:
df.head(5)

,name,position,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,loss_percentage,motm,potm,passing,physical,gk_ability,skill
0,Daco,G,4,7,0.44,0,0,32%,48%,0,0,3.5,5.5,7.5,1.50
1,Leo,D,30,15,1.96,3,3,43%,39%,3,2,7.0,8.5,0.0,4.00
2,Simikj,M,8,19,1.17,0,1,39%,52%,0,0,6.5,3.5,0.0,4.25
3,Stef,A,17,10,1.23,2,0,36%,45%,1,0,5.0,6.0,0.0,2.50
4,Neno,G,0,6,0.27,0,0,36%,41%,0,0,5.0,7.5,6.5,2.00


In [307]:
df['w_percentage'] = df['w_percentage'].str.replace('%','')

In [308]:
df['w_percentage'] = df['w_percentage'].astype(float)/100

In [309]:
df['w_percentage'] = df['w_percentage'].round(decimals=2)

In [310]:
df.head(10)

,name,position,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,loss_percentage,motm,potm,passing,physical,gk_ability,skill
0,Daco,G,4,7,0.44,0,0,0.32,48%,0,0,3.5,5.5,7.5,1.50
1,Leo,D,30,15,1.96,3,3,0.43,39%,3,2,7.0,8.5,0.0,4.00
2,Simikj,M,8,19,1.17,0,1,0.39,52%,0,0,6.5,3.5,0.0,4.25
3,Stef,A,17,10,1.23,2,0,0.36,45%,1,0,5.0,6.0,0.0,2.50
4,Neno,G,0,6,0.27,0,0,0.36,41%,0,0,5.0,7.5,6.5,2.00
5,Premcho,A,26,16,2.21,5,0,0.32,58%,1,0,7.5,7.0,0.0,4.50
6,Hito,A,48,15,3.50,10,2,0.50,33%,4,1,7.0,8.0,0.0,4.50
7,Borjan,M,21,12,1.90,3,0,0.59,35%,0,0,5.0,6.5,0.0,1.50
8,Ancho,D,2,0,0.12,0,0,0.41,41%,0,0,1.5,3.5,0.0,1.00
9,Joker,A,18,13,2.07,0,1,0.47,27%,1,0,6.5,4.0,0.0,3.50


In [311]:
df['loss_percentage'] = df['loss_percentage'].str.replace('%', '')
df['loss_percentage'] = df['loss_percentage'].astype(float) / 100
df['loss_percentage'] = df['loss_percentage'].round(decimals=2)

In [312]:
df.head(10)

,name,position,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,loss_percentage,motm,potm,passing,physical,gk_ability,skill
0,Daco,G,4,7,0.44,0,0,0.32,0.48,0,0,3.5,5.5,7.5,1.50
1,Leo,D,30,15,1.96,3,3,0.43,0.39,3,2,7.0,8.5,0.0,4.00
2,Simikj,M,8,19,1.17,0,1,0.39,0.52,0,0,6.5,3.5,0.0,4.25
3,Stef,A,17,10,1.23,2,0,0.36,0.45,1,0,5.0,6.0,0.0,2.50
4,Neno,G,0,6,0.27,0,0,0.36,0.41,0,0,5.0,7.5,6.5,2.00
5,Premcho,A,26,16,2.21,5,0,0.32,0.58,1,0,7.5,7.0,0.0,4.50
6,Hito,A,48,15,3.50,10,2,0.50,0.33,4,1,7.0,8.0,0.0,4.50
7,Borjan,M,21,12,1.90,3,0,0.59,0.35,0,0,5.0,6.5,0.0,1.50
8,Ancho,D,2,0,0.12,0,0,0.41,0.41,0,0,1.5,3.5,0.0,1.00
9,Joker,A,18,13,2.07,0,1,0.47,0.27,1,0,6.5,4.0,0.0,3.50


In [313]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   name               38 non-null     object 
 1   position           38 non-null     object 
 2   goals              38 non-null     int64  
 3   assists            38 non-null     int64  
 4   gc_per_match       38 non-null     float64
 5   hat-trick          38 non-null     int64  
 6   hat-trick-assists  38 non-null     int64  
 7   w_percentage       38 non-null     float64
 8   loss_percentage    38 non-null     float64
 9   motm               38 non-null     int64  
 10  potm               38 non-null     int64  
 11  passing            38 non-null     float64
 12  physical           38 non-null     float64
 13  gk_ability         38 non-null     float64
 14  skill              38 non-null     float64
dtypes: float64(7), int64(6), object(2)
memory usage: 4.6+ KB


## Attackers
The formula and the stats are set depending on the performance and on the attributes I think football is requiring for an ATTACKER to be complete.
I add more bias to the goals assists, to the physicality, to the skill and to passing

In [314]:
attackers = df[df['position'] =='A']

In [315]:
attackers

,name,position,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,loss_percentage,motm,potm,passing,physical,gk_ability,skill
3,Stef,A,17,10,1.23,2,0,0.36,0.45,1,0,5.0,6.0,0.0,2.5
5,Premcho,A,26,16,2.21,5,0,0.32,0.58,1,0,7.5,7.0,0.0,4.5
6,Hito,A,48,15,3.50,10,2,0.50,0.33,4,1,7.0,8.0,0.0,4.5
9,Joker,A,18,13,2.07,0,1,0.47,0.27,1,0,6.5,4.0,0.0,3.5
14,Lazo,A,7,5,1.50,0,0,0.25,0.25,0,0,4.0,8.0,0.0,2.5
16,Dare,A,9,8,2.43,1,1,0.43,0.43,3,0,8.5,5.0,0.0,5.0
20,Ognen,A,4,3,1.75,0,0,0.25,0.75,0,0,6.5,5.5,0.0,4.5
22,Stojka,A,1,0,0.25,0,0,0.50,0.50,0,0,7.0,2.0,0.0,3.0
25,Vule,A,3,5,4.00,0,1,1.00,0.00,2,0,8.0,8.0,0.0,5.0
26,Nikec,A,1,2,1.50,0,0,0.50,0.00,0,0,5.5,5.5,0.0,2.5


In [316]:
weights_attack = {
    'goals': 0.6, 
    'assists': 0.4,
    'gc_per_match': 0.3, #prop included
    'passing': 0.2, #prop included
    'physical': 0.25, #prop included
    'motm': 1,  
    'potm': 3,  
    'w_percentage': 10,  
    'loss_percentage': 5,
    'win_loss':0.1, #prop included
    'skill':0.15 #prop included
}

attackers = attackers.assign(
    rating=(
        ((attackers['goals'] * weights_attack['goals'] +
          attackers['assists'] * weights_attack['assists']) * attackers['gc_per_match'] * weights_attack['gc_per_match']) +
        (attackers['passing'] * weights_attack['passing']) +
        (attackers['physical'] * weights_attack['physical']) +
        ((attackers['motm'] * weights_attack['motm'] + attackers['potm'] * weights_attack['potm'] +
          attackers['w_percentage'] * weights_attack['w_percentage'] - attackers['loss_percentage'] * weights_attack['loss_percentage']) * weights_attack['win_loss']) + 
        attackers['skill'] * (2 + weights_attack['skill'] )
    )
)


In [317]:
attackers = attackers.sort_values(by='rating', ascending=False)

In [318]:
attackers[['name','rating']]

,name,rating
6,Hito,50.6500
5,Premcho,27.6410
16,Dare,20.4844
9,Joker,20.1960
25,Vule,20.1100
20,Ognen,14.1150
3,Stef,13.3498
33,Tomas,12.3950
14,Lazo,11.0900
28,Teo,10.0500


In [319]:
# scaler_a = MinMaxScaler(feature_range=(1,10))
# scaler_a.fit(attackers[['rating']])
# attackers['rating'] = scaler_a.fit_transform(attackers[['rating']])
# attackers

In [320]:
attackers['rating'] = attackers['rating'].apply(lambda x: x*1.2 if x < 5 else x)

In [321]:
attackers[['name','rating']].round(3)

,name,rating
6,Hito,50.650
5,Premcho,27.641
16,Dare,20.484
9,Joker,20.196
25,Vule,20.110
20,Ognen,14.115
3,Stef,13.350
33,Tomas,12.395
14,Lazo,11.090
28,Teo,10.050


In [322]:
avg_attackers = round((attackers['rating'].sum())/attackers['rating'].size, 3)

In [323]:
avg_attackers

16.62

## Midfield

In [324]:
midfielders = df[df['position'] =='M']

In [325]:
midfielders

,name,position,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,loss_percentage,motm,potm,passing,physical,gk_ability,skill
2,Simikj,M,8,19,1.17,0,1,0.39,0.52,0,0,6.5,3.5,0.0,4.25
7,Borjan,M,21,12,1.90,3,0,0.59,0.35,0,0,5.0,6.5,0.0,1.50
17,Mario,M,8,5,2.17,0,0,0.67,0.17,3,0,6.0,8.0,0.0,4.00
18,Martin,M,7,5,2.40,2,0,0.20,0.60,3,0,8.0,8.5,0.0,4.50
23,Jancho,M,3,5,2.67,0,1,0.33,0.33,0,0,6.0,4.0,0.0,2.50
24,Davide,M,2,3,1.67,0,0,0.00,0.67,0,0,7.5,7.5,0.0,5.00
30,Theo,M,4,2,6.00,1,0,1.00,0.00,0,0,6.5,5.0,0.0,3.50
37,Leon,M,0,0,0.00,0,0,0.00,0.00,0,0,4.5,6.0,0.0,2.00


In [326]:
weights_midfielders = {
    'goals': 0.5, 
    'assists': 0.5,
    'gc_per_match': 0.25,#prop included
    'passing': 0.3, #prop included
    'physical': 0.2, #prop included
    'motm': 1,  
    'potm': 3,  
    'w_percentage': 10,  
    'loss_percentage': 5,
    'win_loss':0.1, #prop included
    'skill':0.15 #prop included
}

midfielders = midfielders.assign(
    rating=(
        ((midfielders['goals'] * weights_midfielders['goals'] +
          midfielders['assists'] * weights_midfielders['assists']) * midfielders['gc_per_match'] * weights_midfielders['gc_per_match']) +
        (midfielders['passing'] * weights_midfielders['passing']) +
        (midfielders['physical'] * weights_midfielders['physical']) +
        ((midfielders['motm'] * weights_midfielders['motm'] + midfielders['potm'] * weights_midfielders['potm'] +
          midfielders['w_percentage'] * weights_midfielders['w_percentage'] - midfielders['loss_percentage'] * weights_midfielders['loss_percentage']) * weights_midfielders['win_loss']) + 
        midfielders['skill'] * (3 + weights_midfielders['skill'] )
    )
)


In [327]:
midfielders = midfielders.sort_values(by='rating', ascending=False)

In [328]:
midfielders[['name','rating']].round(3)

,name,rating
18,Martin,22.075
17,Mario,20.411
24,Davide,20.209
2,Simikj,20.116
30,Theo,19.475
7,Borjan,15.777
23,Jancho,13.310
37,Leon,8.850


In [329]:
# scaler_m = MinMaxScaler(feature_range=(1,10))
# scaler_m.fit(midfielders[['rating']])
# midfielders['rating'] = scaler_m.fit_transform(midfielders[['rating']])

In [330]:
midfielders[['name','rating']].round(3)

,name,rating
18,Martin,22.075
17,Mario,20.411
24,Davide,20.209
2,Simikj,20.116
30,Theo,19.475
7,Borjan,15.777
23,Jancho,13.310
37,Leon,8.850


In [331]:
# midfielders['rating'] = midfielders['rating'].apply(lambda x: x*1.2 if x < 5 else x)

In [332]:
midfielders[['name','rating']].round(3)

,name,rating
18,Martin,22.075
17,Mario,20.411
24,Davide,20.209
2,Simikj,20.116
30,Theo,19.475
7,Borjan,15.777
23,Jancho,13.310
37,Leon,8.850


In [333]:
avg_midfielders = round((midfielders['rating'].sum())/midfielders['rating'].size, 3)

In [334]:
avg_midfielders

17.528

## Defenders

In [335]:
defenders = df[df['position'] =='D']

In [336]:
defenders

,name,position,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,loss_percentage,motm,potm,passing,physical,gk_ability,skill
1,Leo,D,30,15,1.96,3,3,0.43,0.39,3,2,7.0,8.5,0.0,4.0
8,Ancho,D,2,0,0.12,0,0,0.41,0.41,0,0,1.5,3.5,0.0,1.0
10,Shulz,D,8,11,1.19,0,1,0.50,0.25,1,1,6.5,8.5,0.0,3.0
11,Maki,D,4,11,1.00,0,1,0.53,0.33,2,1,5.5,6.5,0.0,4.0
12,Grekos,D,7,3,0.71,0,0,0.21,0.57,0,0,4.0,5.5,0.0,2.5
13,Bale,D,1,2,0.30,0,0,0.30,0.40,1,0,2.0,9.0,0.0,1.0
34,Tagar,D,1,0,1.00,0,0,1.00,0.00,0,0,3.0,7.0,0.0,1.0
35,Gorjan,D,0,0,0.00,0,0,0.00,1.00,0,0,3.5,8.0,0.0,1.5
36,Hris,D,0,0,0.00,0,0,0.00,1.00,0,0,2.0,5.0,0.0,1.0


In [337]:
weight_defenders = {
    'goals': 0.5, 
    'assists': 0.5,
    'gc_per_match': 0.25,#prop included
    'passing': 0.27, #prop included
    'physical': 0.25, #prop included
    'motm': 1,  
    'potm': 3,  
    'w_percentage': 10,  
    'loss_percentage': 5,
    'win_loss':0.1, #prop included
    'skill':0.13 #prop included
}

defenders = defenders.assign(
    rating=(
        ((defenders['goals'] * weight_defenders['goals'] +
          defenders['assists'] * weight_defenders['assists']) * defenders['gc_per_match'] * weight_defenders['gc_per_match']) +
        (defenders['passing'] * weight_defenders['passing']) +
        (defenders['physical'] * weight_defenders['physical']) +
        ((defenders['motm'] * weight_defenders['motm'] + defenders['potm'] * weight_defenders['potm'] +
          defenders['w_percentage'] * weight_defenders['w_percentage'] - defenders['loss_percentage'] * weight_defenders['loss_percentage']) * weight_defenders['win_loss']) + 
        defenders['skill'] * (3 + weight_defenders['skill'] )
    )
)


In [338]:
defenders = defenders.sort_values(by='rating', ascending=False)

In [339]:
defenders[['name','rating']].round(3)

,name,rating
1,Leo,28.695
11,Maki,18.370
10,Shulz,16.871
12,Grekos,11.092
35,Gorjan,7.140
34,Tagar,6.815
13,Bale,6.232
8,Ancho,4.645
36,Hris,4.420


In [340]:
# scaler_d = MinMaxScaler(feature_range=(1,10))
# scaler_d.fit(defenders[['rating']])
# defenders['rating'] = scaler_d.fit_transform(defenders[['rating']])

In [341]:
avg_defenders = round((defenders['rating'].sum())/defenders['rating'].size, 3)

In [342]:
avg_defenders

11.587

In [343]:
# defenders['rating'] = defenders['rating'].apply(lambda x: x*1.2 if x < 5 else x)

In [344]:
defenders[['name','rating']].round(3)

,name,rating
1,Leo,28.695
11,Maki,18.370
10,Shulz,16.871
12,Grekos,11.092
35,Gorjan,7.140
34,Tagar,6.815
13,Bale,6.232
8,Ancho,4.645
36,Hris,4.420


## Goalkeepers

In [345]:
goalkeepers = df[df['position'] =='G']

In [346]:
goalkeepers

,name,position,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,loss_percentage,motm,potm,passing,physical,gk_ability,skill
0,Daco,G,4,7,0.44,0,0,0.32,0.48,0,0,3.5,5.5,7.5,1.5
4,Neno,G,0,6,0.27,0,0,0.36,0.41,0,0,5.0,7.5,6.5,2.0
15,Stec,G,0,1,0.11,0,0,0.67,0.22,2,0,1.5,1.0,6.5,1.0
19,Fich,G,1,0,0.20,0,0,0.20,0.80,0,0,4.0,6.0,6.0,2.0
21,Nuh,G,0,1,0.25,0,0,0.50,0.50,0,0,1.5,3.0,5.0,1.0
27,Fish,G,0,0,0.00,0,0,0.50,0.50,0,0,1.0,1.5,2.0,1.0
29,Silbo,G,0,0,0.00,0,0,0.50,0.00,1,0,3.5,7.5,6.0,1.0


In [347]:
goalkeepers[['name','gk_ability']].round(3)

,name,gk_ability
0,Daco,7.5
4,Neno,6.5
15,Stec,6.5
19,Fich,6.0
21,Nuh,5.0
27,Fish,2.0
29,Silbo,6.0


In [348]:
weights_goalkeepers = {
    'gk_ability': 0.6, #prop included
    'passing': 0.1, #prop included
    'physical': 0.1, #prop included
    'goal_contribution':0.1,
    'motm': 2,  
    'potm': 4,  
    'w_percentage': 10,  
    'loss_percentage': 5,
    'win_loss':0.1, #prop included
}

goalkeepers = goalkeepers.assign(
    rating=(
        (goalkeepers['goals'] + goalkeepers['assists']) * weights_goalkeepers['goal_contribution'] + 
        goalkeepers['gk_ability'] * 2 +
        (goalkeepers['w_percentage'] * weights_goalkeepers['w_percentage'] + goalkeepers['loss_percentage'] * weights_goalkeepers['loss_percentage'])*weights_goalkeepers['win_loss'] +
        goalkeepers['passing'] * weights_goalkeepers['passing'] + goalkeepers['physical'] * weights_goalkeepers['physical'] +
        goalkeepers['motm'] * weights_goalkeepers['motm'] +
        goalkeepers['potm'] * weights_goalkeepers['potm']
    )
)


In [349]:
goalkeepers = goalkeepers.sort_values(by='rating', ascending=False)

In [350]:
# scaler_gk = MinMaxScaler(feature_range=(1,10))
# scaler_gk.fit(goalkeepers[['rating']])
# goalkeepers['rating'] = scaler_gk.fit_transform(goalkeepers[['rating']])

In [351]:
# goalkeepers['rating'] = goalkeepers['rating'].apply(lambda x: x*1.2 if x < 5 else x)

In [352]:
goalkeepers[['name','rating']].round(3)

,name,rating
15,Stec,18.130
0,Daco,17.560
29,Silbo,15.600
4,Neno,15.415
19,Fich,13.700
21,Nuh,11.300
27,Fish,5.000


## Merging the ratings

In [353]:
df_with_ratings = pd.concat([goalkeepers,defenders,midfielders,attackers], ignore_index=True)

In [354]:
df_with_ratings

,name,position,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,loss_percentage,motm,potm,passing,physical,gk_ability,skill,rating
0,Stec,G,0,1,0.11,0,0,0.67,0.22,2,0,1.5,1.0,6.5,1.00,18.13000
1,Daco,G,4,7,0.44,0,0,0.32,0.48,0,0,3.5,5.5,7.5,1.50,17.56000
2,Silbo,G,0,0,0.00,0,0,0.50,0.00,1,0,3.5,7.5,6.0,1.00,15.60000
3,Neno,G,0,6,0.27,0,0,0.36,0.41,0,0,5.0,7.5,6.5,2.00,15.41500
4,Fich,G,1,0,0.20,0,0,0.20,0.80,0,0,4.0,6.0,6.0,2.00,13.70000
5,Nuh,G,0,1,0.25,0,0,0.50,0.50,0,0,1.5,3.0,5.0,1.00,11.30000
6,Fish,G,0,0,0.00,0,0,0.50,0.50,0,0,1.0,1.5,2.0,1.00,5.00000
7,Leo,D,30,15,1.96,3,3,0.43,0.39,3,2,7.0,8.5,0.0,4.00,28.69500
8,Maki,D,4,11,1.00,0,1,0.53,0.33,2,1,5.5,6.5,0.0,4.00,18.37000
9,Shulz,D,8,11,1.19,0,1,0.50,0.25,1,1,6.5,8.5,0.0,3.00,16.87125


In [355]:
# scaler = MinMaxScaler(feature_range=(1,10))
# scaler.fit(df_with_ratings[['rating']])
# df_with_ratings['rating'] = scaler.fit_transform(df_with_ratings[['rating']])

In [356]:
df_with_ratings = df_with_ratings[['name','rating']].round(3)

In [376]:
df_with_ratings.sort_values(by='rating',ascending=False)

,name,rating
24,Hito,50.650
7,Leo,28.695
25,Premcho,27.641
16,Martin,22.075
26,Dare,20.484
17,Mario,20.411
18,Davide,20.209
27,Joker,20.196
19,Simikj,20.116
28,Vule,20.110


## Selected players

In [357]:
selected_players = ['Stec','Neno','Vule','Hito','Borjan','Premcho','Leo','Shulz','Simikj','Stef','Ancho','Maki']

In [358]:
match_players_df = df_with_ratings[df_with_ratings['name'].isin(selected_players)]

In [359]:
match_players_df

,name,rating
0,Stec,18.130
3,Neno,15.415
7,Leo,28.695
8,Maki,18.370
9,Shulz,16.871
14,Ancho,4.645
19,Simikj,20.116
21,Borjan,15.777
24,Hito,50.650
25,Premcho,27.641


In [360]:
match_gks = match_players_df[match_players_df['position'] == 'G']
match_outfield = match_players_df[match_players_df['position'] != 'G']

KeyError: 'position'

In [ ]:
if len(match_gks) < 2: 
    raise ValueError("Need at least two goalkeepers!")
#TODO if no goalkeepers for the match

In [361]:
match_gks = match_gks.sort_values(by='rating', ascending=False).reset_index(drop=True)
match_outfield = match_outfield.sort_values(by='rating', ascending=False).reset_index(drop=True)

In [362]:
match_outfield

,name,position,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,loss_percentage,motm,potm,passing,physical,gk_ability,skill,rating
0,Hito,A,48,15,3.50,10,2,0.50,0.33,4,1,7.0,8.0,0.0,4.50,50.65000
1,Leo,D,30,15,1.96,3,3,0.43,0.39,3,2,7.0,8.5,0.0,4.00,28.69500
2,Premcho,A,26,16,2.21,5,0,0.32,0.58,1,0,7.5,7.0,0.0,4.50,27.64100
3,Simikj,M,8,19,1.17,0,1,0.39,0.52,0,0,6.5,3.5,0.0,4.25,20.11625
4,Vule,A,3,5,4.00,0,1,1.00,0.00,2,0,8.0,8.0,0.0,5.00,20.11000
5,Maki,D,4,11,1.00,0,1,0.53,0.33,2,1,5.5,6.5,0.0,4.00,18.37000
6,Shulz,D,8,11,1.19,0,1,0.50,0.25,1,1,6.5,8.5,0.0,3.00,16.87125
7,Borjan,M,21,12,1.90,3,0,0.59,0.35,0,0,5.0,6.5,0.0,1.50,15.77750
8,Stef,A,17,10,1.23,2,0,0.36,0.45,1,0,5.0,6.0,0.0,2.50,13.34980
9,Ancho,D,2,0,0.12,0,0,0.41,0.41,0,0,1.5,3.5,0.0,1.00,4.64500


In [363]:
match_gks

,name,position,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,loss_percentage,motm,potm,passing,physical,gk_ability,skill,rating
0,Stec,G,0,1,0.11,0,0,0.67,0.22,2,0,1.5,1.0,6.5,1.0,18.130
1,Neno,G,0,6,0.27,0,0,0.36,0.41,0,0,5.0,7.5,6.5,2.0,15.415


In [364]:
#Snake algorithm
team_a = []
team_b = []
for i in range(0, len(match_outfield), 4):
    picks = match_outfield.iloc[i:i+4]
    if len(picks) >= 4:
        team_a += [picks.iloc[0], picks.iloc[3]]
        team_b += [picks.iloc[1], picks.iloc[2]]
    else:
        for j, row in picks.iterrows():
            if len(team_a) <= len(team_b):
                team_a.append(row)
            else:
                team_b.append(row)

In [365]:
team_a = pd.DataFrame(team_a)
team_b = pd.DataFrame(team_b)

In [366]:
team_a

,name,position,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,loss_percentage,motm,potm,passing,physical,gk_ability,skill,rating
0,Hito,A,48,15,3.50,10,2,0.50,0.33,4,1,7.0,8.0,0.0,4.50,50.65000
3,Simikj,M,8,19,1.17,0,1,0.39,0.52,0,0,6.5,3.5,0.0,4.25,20.11625
4,Vule,A,3,5,4.00,0,1,1.00,0.00,2,0,8.0,8.0,0.0,5.00,20.11000
7,Borjan,M,21,12,1.90,3,0,0.59,0.35,0,0,5.0,6.5,0.0,1.50,15.77750
8,Stef,A,17,10,1.23,2,0,0.36,0.45,1,0,5.0,6.0,0.0,2.50,13.34980


In [367]:
team_b

,name,position,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,loss_percentage,motm,potm,passing,physical,gk_ability,skill,rating
1,Leo,D,30,15,1.96,3,3,0.43,0.39,3,2,7.0,8.5,0.0,4.0,28.69500
2,Premcho,A,26,16,2.21,5,0,0.32,0.58,1,0,7.5,7.0,0.0,4.5,27.64100
5,Maki,D,4,11,1.00,0,1,0.53,0.33,2,1,5.5,6.5,0.0,4.0,18.37000
6,Shulz,D,8,11,1.19,0,1,0.50,0.25,1,1,6.5,8.5,0.0,3.0,16.87125
9,Ancho,D,2,0,0.12,0,0,0.41,0.41,0,0,1.5,3.5,0.0,1.0,4.64500


In [368]:
rating_team_a = team_a['rating'].sum()
rating_team_b = team_b['rating'].sum()

In [369]:
if rating_team_a < rating_team_b:
    team_a = pd.concat([pd.DataFrame([goalkeepers.iloc[0]]), team_a], ignore_index=True)
    team_b = pd.concat([pd.DataFrame([goalkeepers.iloc[1]]), team_b], ignore_index=True)
else:
    team_b = pd.concat([pd.DataFrame([goalkeepers.iloc[0]]), team_b], ignore_index=True)
    team_a = pd.concat([pd.DataFrame([goalkeepers.iloc[1]]), team_a], ignore_index=True)

In [370]:
team_a

,name,position,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,loss_percentage,motm,potm,passing,physical,gk_ability,skill,rating
0,Daco,G,4,7,0.44,0,0,0.32,0.48,0,0,3.5,5.5,7.5,1.50,17.56000
1,Hito,A,48,15,3.50,10,2,0.50,0.33,4,1,7.0,8.0,0.0,4.50,50.65000
2,Simikj,M,8,19,1.17,0,1,0.39,0.52,0,0,6.5,3.5,0.0,4.25,20.11625
3,Vule,A,3,5,4.00,0,1,1.00,0.00,2,0,8.0,8.0,0.0,5.00,20.11000
4,Borjan,M,21,12,1.90,3,0,0.59,0.35,0,0,5.0,6.5,0.0,1.50,15.77750
5,Stef,A,17,10,1.23,2,0,0.36,0.45,1,0,5.0,6.0,0.0,2.50,13.34980


In [371]:
team_b

,name,position,goals,assists,gc_per_match,hat-trick,hat-trick-assists,w_percentage,loss_percentage,motm,potm,passing,physical,gk_ability,skill,rating
0,Stec,G,0,1,0.11,0,0,0.67,0.22,2,0,1.5,1.0,6.5,1.0,18.13000
1,Leo,D,30,15,1.96,3,3,0.43,0.39,3,2,7.0,8.5,0.0,4.0,28.69500
2,Premcho,A,26,16,2.21,5,0,0.32,0.58,1,0,7.5,7.0,0.0,4.5,27.64100
3,Maki,D,4,11,1.00,0,1,0.53,0.33,2,1,5.5,6.5,0.0,4.0,18.37000
4,Shulz,D,8,11,1.19,0,1,0.50,0.25,1,1,6.5,8.5,0.0,3.0,16.87125
5,Ancho,D,2,0,0.12,0,0,0.41,0.41,0,0,1.5,3.5,0.0,1.0,4.64500


In [372]:
names_team_a = team_a['name'].tolist()
names_team_b = team_b['name'].tolist()


In [373]:
names_team_a

['Daco', 'Hito', 'Simikj', 'Vule', 'Borjan', 'Stef']

In [374]:
names_team_b

['Stec', 'Leo', 'Premcho', 'Maki', 'Shulz', 'Ancho']

In [374]:
#TODO